### Importing Logging and SmartConnect

In [1]:
from SmartApi import SmartConnect
from SmartApi.loggerConfig import get_logger 

logger = get_logger(__name__, "INFO")

#### Define client info

In [ ]:
client_info = {
    "api_key": "",
    "client_id": "",
    "password": "",
    "totp_secret": "",
}

#### Generate SmartApi session

In [ ]:
import pyotp

try:
    totp = pyotp.TOTP(client_info["totp_secret"]).now()
except Exception as e:
    logger.error("Invalid Token: The provided token is not valid.")
    raise e

api = SmartConnect(api_key=client_info["api_key"])

response  = api.generateSession(client_info["client_id"], client_info["password"], totp)
response

### Historical Data

##### [SmartAPI Historical API Documentation](https://smartapi.angelbroking.com/docs/Historical)

#### Fetch Candle Data

In [23]:
try:
    exchange = "NSE"
    symbol_token = "3045"
    interval = "ONE_MINUTE" # or "FIVE_MINUTE" ect
    fromdate_str = "2025-08-10 09:15"
    todate_str =  "2025-09-10 15:15"
    
    candleParams = {
        "exchange": exchange,
        "symboltoken": symbol_token,
        "interval": interval,
        "fromdate": fromdate_str,
        "todate": todate_str
    }
    candledetails = api.getCandleData(candleParams)
    if candledetails["status"]:
        candle_data = candledetails["data"]
        logger.info("Historical Candle Data fetched successfully")
    
except Exception as e:
    logger.exception(f"Failed to fetch candle data: {e}")

2025-09-06 12:27:42,833 - __main__ - INFO - Historical Candle Data fetched successfully


#### Convert Candle Data to DataFrame

In [24]:
import pandas as pd

DEFAULT_DATAFRAME_COLUMNS = ["datetime", "open", "high", "low", "close", "volume"]

try:
    if candle_data:
        candle_df = pd.DataFrame(candle_data, columns=DEFAULT_DATAFRAME_COLUMNS)
        candle_df['datetime'] = pd.to_datetime(candle_df['datetime'])
        logger.info("Converted candle data to DataFrame")
        
    else:
        candle_df = pd.DataFrame()
        logger.warning("No candle data to convert")
        
except Exception as e:
    logger.exception(f"Failed to convert candle data: {e}")
    candle_df = pd.DataFrame()

2025-09-06 12:27:44,586 - __main__ - INFO - Converted candle data to DataFrame


In [25]:
candle_df

,datetime,open,high,low,close,volume
0,2025-08-11 09:15:00+05:30,808.00,812.40,808.00,811.45,298663
1,2025-08-11 09:16:00+05:30,811.45,813.80,811.35,812.95,304812
2,2025-08-11 09:17:00+05:30,813.10,815.80,812.00,815.70,303040
3,2025-08-11 09:18:00+05:30,815.70,816.80,815.25,816.50,165810
4,2025-08-11 09:19:00+05:30,816.20,816.90,814.55,816.00,131647
...,...,...,...,...,...,...
6789,2025-09-06 12:23:00+05:30,810.70,820.00,810.00,814.00,21713
6790,2025-09-06 12:24:00+05:30,815.00,828.35,812.10,826.05,142693
6791,2025-09-06 12:25:00+05:30,826.05,827.10,812.00,821.10,38552
6792,2025-09-06 12:26:00+05:30,820.10,821.10,815.00,820.10,43140


#### Save DataFrame to CSV

In [26]:
try:
    if not candle_df.empty:
        csv_filename = "candle_data.csv"
        candle_df.to_csv(csv_filename, index=False)
        logger.info(f"Candle data saved to CSV file: {csv_filename}")
    else:
        logger.warning("DataFrame is empty, nothing to save")
except Exception as e:
    logger.exception(f"Failed to save candle data to CSV: {e}")

2025-09-06 12:31:16,605 - __main__ - INFO - Candle data saved to CSV file: candle_data.csv
